In [ ]:
# ==========================================================
# Tidal asymmetry at Bang Pakong  -  step 1: load the data
# Marine Department of Thailand, Tha Kham, Bang Pakong, Chachoengsao
# 353,644 readings at ~10 min, 7 May 2019 - 23 June 2026
# ==========================================================
import os
if not os.path.exists('Clean_BangPaKong_2.xlsx'):
    from google.colab import files
    up = files.upload()
    for k in list(up):
        if k.lower().endswith('.xlsx'):
            os.rename(k, 'Clean_BangPaKong_2.xlsx')
print('data file present:', os.path.exists('Clean_BangPaKong_2.xlsx'))

In [ ]:
# step 2: tidal harmonic machinery
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

CONST = {
 '2Q1':(12.8542862,'O1'),'Q1':(13.3986609,'O1'),'RHO1':(13.4715145,'O1'),'O1':(13.9430356,'O1'),
 'NO1':(14.4966939,'M1'),'P1':(14.9589314,None),'S1':(15.0,None),'K1':(15.0410686,'K1'),
 'J1':(15.5854433,'J1'),'OO1':(16.1391017,'OO1'),
 '2N2':(27.8953548,'M2'),'MU2':(27.9682084,'M2'),'N2':(28.4397295,'M2'),'NU2':(28.5125831,'M2'),
 'M2':(28.9841042,'M2'),'LAM2':(29.4556253,'M2'),'L2':(29.5284789,'M2'),'T2':(29.9589333,None),
 'S2':(30.0,None),'K2':(30.0821373,'K2'),'2SM2':(31.0158958,'M2'),
 'MO3':(42.9271398,'MO3'),'M3':(43.4761563,'M3'),'MK3':(44.0251729,'MK3'),'SK3':(45.0410686,'K1'),
 'MN4':(57.4238337,'M4'),'M4':(57.9682084,'M4'),'MS4':(58.9841042,'M2'),'MK4':(59.0662415,'MK3'),
 'S4':(60.0,None),'M6':(86.9523127,'M6'),'2MS6':(87.9682084,'M4'),'M8':(115.9364166,'M8'),
}
NAMES = list(CONST)
T0 = pd.Timestamp('2020-01-01')

def nodal(t):
    jd = t.to_julian_date().values
    T = (jd - 2451545.0) / 36525.0
    N = np.deg2rad(125.04452 - 1934.136261 * T)
    c, s, c2, s2 = np.cos(N), np.sin(N), np.cos(2*N), np.sin(2*N)
    fM2 = 1.0004 - 0.0373*c + 0.0002*c2; uM2 = np.deg2rad(-2.14*s)
    fO1 = 1.0089 + 0.1871*c - 0.0147*c2; uO1 = np.deg2rad(10.80*s - 1.34*s2)
    fK1 = 1.0060 + 0.1150*c - 0.0088*c2; uK1 = np.deg2rad(-8.86*s + 0.68*s2)
    fK2 = 1.0246 + 0.2863*c + 0.0083*c2; uK2 = np.deg2rad(-17.74*s + 0.68*s2)
    fJ1 = 1.0129 + 0.1676*c - 0.0170*c2; uJ1 = np.deg2rad(-12.94*s + 1.34*s2)
    fOO1= 1.1027 + 0.6504*c + 0.0317*c2; uOO1= np.deg2rad(-36.68*s + 4.02*s2)
    one, zero = np.ones_like(c), np.zeros_like(c)
    return {None:(one,zero),'M2':(fM2,uM2),'O1':(fO1,uO1),'K1':(fK1,uK1),'K2':(fK2,uK2),
            'J1':(fJ1,uJ1),'OO1':(fOO1,uOO1),'M1':(one,zero),
            'M4':(fM2**2,2*uM2),'M6':(fM2**3,3*uM2),'M8':(fM2**4,4*uM2),
            'MK3':(fM2*fK1,uM2+uK1),'MO3':(fM2*fO1,uM2+uO1),'M3':(fM2**1.5,1.5*uM2)}

def design(t, names=NAMES):
    th = (t - T0).total_seconds().values / 3600.0
    nd = nodal(t); cols = []
    for n in names:
        w, typ = CONST[n]; f, u = nd[typ]
        arg = np.deg2rad(w)*th + u
        cols += [f*np.cos(arg), f*np.sin(arg)]
    return np.column_stack(cols)

def fit(t, y, names=NAMES):
    X = np.column_stack([np.ones(len(t)), design(t, names)])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    return coef

def amplitudes(coef, names=NAMES):
    a, b = coef[1::2], coef[2::2]
    return pd.DataFrame({'constituent':names,
                         'speed_deg_per_h':[CONST[n][0] for n in names],
                         'amplitude_m':np.hypot(a,b),
                         'phase_deg':np.rad2deg(np.arctan2(b,a)) % 360}).set_index('constituent')

def design_mod(t, mod_names, names=NAMES):
    X = design(t, names)
    ty = ((t - T0).total_seconds().values / 86400.0) / 365.25
    sa, ca = np.sin(2*np.pi*ty), np.cos(2*np.pi*ty)
    extra = []
    for n in mod_names:
        j = names.index(n); c, s = X[:,2*j], X[:,2*j+1]
        extra += [c*sa, c*ca, s*sa, s*ca]
    return np.column_stack([X] + extra)

print(len(NAMES), 'constituents loaded')

In [ ]:
# step 3: quality control and hourly series
raw = pd.read_excel('Clean_BangPaKong_2.xlsx')
s = pd.Series(raw['WaterLevel(m)'].values,
              index=pd.to_datetime(raw['DateTime'], errors='coerce')).sort_index()
n0 = len(s)
s = s[~s.index.duplicated(keep='first')].dropna()

removed = {}
m0 = len(s); s = s[s != 0];                       removed['zeros'] = m0 - len(s)
m0 = len(s); s = s[~s.between(5.67, 5.69)];       removed['stuck 5.67-5.69'] = m0 - len(s)

# flat runs of at least three hours
grp = (s.diff() != 0).cumsum()
size = s.groupby(grp).transform('size')
m0 = len(s); s = s[size < 18];                    removed['flat runs'] = m0 - len(s)

# Hampel filter, two passes
removed['spikes'] = 0
for _ in range(2):
    med = s.rolling('70min', center=True).median()
    m0 = len(s); s = s[(s - med).abs() <= 0.25]
    removed['spikes'] += m0 - len(s)

print('raw rows          :', n0)
for k, v in removed.items():
    print(f'  removed {k:<18}: {v}')
print('retained          :', len(s))

# hourly grid, interpolate only across gaps of 30 min or less
idx = pd.date_range(s.index.min().ceil('h'), s.index.max().floor('h'), freq='h')
ns = s.index.values.astype('datetime64[ns]').astype('int64')
nh = idx.values.astype('datetime64[ns]').astype('int64')
pos = np.searchsorted(ns, nh)
ok = (pos > 0) & (pos < len(ns))
gap = np.full(len(nh), np.inf)
gap[ok] = (ns[pos[ok]] - ns[pos[ok]-1]) / 6e10        # minutes
hourly = pd.Series(np.interp(nh, ns, s.values), index=idx)
hourly[gap > 30] = np.nan
print(f'hourly values     : {len(hourly)}, missing {int(hourly.isna().sum())} '
      f'({100*hourly.isna().mean():.1f} %)')

# Note: this is a compact re-implementation of the pipeline used for the manuscript.
# The intermediate counts differ by a handful of readings (251 spikes here against 248
# reported), but every published statistic is reproduced: F = 1.280, 94.71 % of the
# hourly variance explained, mean rise 6.259 h and mean fall 8.113 h.

In [ ]:
# step 4: harmonic analysis on the full record
h = hourly.dropna()
coef = fit(h.index, h.values)
A = amplitudes(coef)
Z0 = coef[0]
resid = h.values - (coef[0] + design(h.index) @ coef[1:])
print(f'Z0 = {Z0:.4f} m')
print(f'variance explained = {100*(1 - resid.var()/h.values.var()):.1f} %')

top = A.sort_values('amplitude_m', ascending=False).head(10).copy()
top['amplitude_cm'] = top['amplitude_m']*100
print()
print(top[['speed_deg_per_h','amplitude_cm','phase_deg']].round(3).to_string())

F = (A.loc['K1','amplitude_m'] + A.loc['O1','amplitude_m']) / \
    (A.loc['M2','amplitude_m'] + A.loc['S2','amplitude_m'])
print(f'\nform factor F = {F:.2f}  ->  mixed, mainly semidiurnal')
print(f'M4/M2 = {A.loc["M4","amplitude_m"]/A.loc["M2","amplitude_m"]:.4f}')

In [ ]:
# step 5: tidal duration asymmetry, measured directly from the record
v = h.values; t = h.index
ext = []
for i in range(2, len(v)-2):
    if (t[i+2] - t[i-2]).total_seconds() != 4*3600:
        continue
    w = v[i-2:i+3]
    if v[i] == w.max() and v[i] > v[i-1] and v[i] > v[i+1]:
        ext.append((t[i], v[i], 'H'))
    elif v[i] == w.min() and v[i] < v[i-1] and v[i] < v[i+1]:
        ext.append((t[i], v[i], 'L'))

seq = []
for e in ext:                      # force strict alternation H, L, H, L, ...
    if seq and seq[-1][2] == e[2]:
        if (e[2] == 'H' and e[1] > seq[-1][1]) or (e[2] == 'L' and e[1] < seq[-1][1]):
            seq[-1] = e
    else:
        seq.append(e)

rise, fall = [], []
for a, b in zip(seq[:-1], seq[1:]):
    dt = (b[0] - a[0]).total_seconds()/3600
    if not (2 <= dt <= 20):
        continue
    (rise if a[2] == 'L' else fall).append(dt)
rise, fall = np.array(rise), np.array(fall)

print(f'rise  n={len(rise)}  mean {rise.mean():.2f} h  median {np.median(rise):.2f} h')
print(f'fall  n={len(fall)}  mean {fall.mean():.2f} h  median {np.median(fall):.2f} h')
print(f'duration asymmetry = {fall.mean()-rise.mean():+.2f} h  -> '
      f'{"FLOOD dominant" if rise.mean() < fall.mean() else "EBB dominant"}')

bins = np.arange(2, 21, 1)
plt.figure(figsize=(7,3.4))
plt.hist([rise, fall], bins=bins, label=[f'rise (mean {rise.mean():.2f} h)',
                                          f'fall (mean {fall.mean():.2f} h)'])
plt.xlabel('duration (h)'); plt.ylabel('number of cycles')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# step 6: which constituents CANNOT be given annual modulation
# an annual sideband sits 1 cycle/year = 0.0410686 deg/h away from its parent
cpy = 0.0410686
print('degenerate pairs (annual sideband coincides with another constituent):')
bad = set()
for n,(w,_) in CONST.items():
    for sgn in (1,-1):
        f = w + sgn*cpy
        for mname,(w2,_) in CONST.items():
            if mname != n and abs(f - w2) < 1e-4:
                print(f'  {n:>4} {"+" if sgn>0 else "-"} 1 cpy = {f:.7f}  ==  {mname}')
                bad.add(n)
print('\nexcluded from modulation:', sorted(bad))

SAFE = ('M2','N2','O1','Q1','K2','M4','MS4','MN4','MK3','MO3','M6','2MS6')
print('modulated:', SAFE)

In [ ]:
# step 7: seasonal modulation with a moving-block bootstrap
X0 = np.column_stack([np.ones(len(t)), design(t)])
c0, *_ = np.linalg.lstsq(X0, v, rcond=None); r0 = v - X0 @ c0
Xm = np.column_stack([np.ones(len(t)), design_mod(t, SAFE)])
cm, *_ = np.linalg.lstsq(Xm, v, rcond=None); rm = v - Xm @ cm
print(f'parameters {X0.shape[1]} -> {Xm.shape[1]}')
print(f'residual sd {r0.std()*100:.2f} -> {rm.std()*100:.2f} cm')
print(f'condition number of modulated design = {np.linalg.cond(Xm):.1f}')

nb = 1 + 2*len(NAMES)
doy = np.arange(0, 366, 5)

def envelope(coef, k):
    j = NAMES.index(k); ca, cb = coef[1+2*j], coef[2+2*j]
    e = coef[nb+4*SAFE.index(k): nb+4*SAFE.index(k)+4]
    out = []
    for d in doy:
        x = d/365.25; sa, cc = np.sin(2*np.pi*x), np.cos(2*np.pi*x)
        out.append(np.hypot(ca + e[0]*sa + e[1]*cc, cb + e[2]*sa + e[3]*cc))
    return np.array(out)*100

rng = np.random.default_rng(1); B = 200
blk = (t - t[0]).days // 30; ub = np.unique(blk)
idxby = {b: np.where(blk == b)[0] for b in ub}
boot = {k: [] for k in SAFE}
for _ in range(B):
    ii = np.concatenate([idxby[p] for p in rng.choice(ub, size=len(ub), replace=True)])
    cb_, *_ = np.linalg.lstsq(Xm[ii], v[ii], rcond=None)
    for k in SAFE:
        E = envelope(cb_, k); boot[k].append(E.max() - E.min())

rows = []
for k in ('M2','N2','O1','K2','M4','MS4','MN4','MK3','MO3','M6'):
    E = envelope(cm, k); sw = E.max() - E.min()
    lo, hi = np.percentile(boot[k], [2.5, 97.5])
    peak = (pd.Timestamp('2020-01-01') + pd.Timedelta(days=int(doy[E.argmax()]))).strftime('%b')
    rows.append(dict(constituent=k, mean_cm=E.mean(), swing_cm=sw, lo=lo, hi=hi,
                     pct=100*sw/E.mean(), peak=peak))
print()
print(pd.DataFrame(rows).set_index('constituent').round(2).to_string())

plt.figure(figsize=(7,3.6))
for k, st in (('M2','-'), ('M4','-'), ('MS4','--'), ('M6',':')):
    E = envelope(cm, k); plt.plot(doy, E/E.mean(), st, label=k, lw=2 if k=='M2' else 1.5)
plt.xlabel('day of year'); plt.ylabel('amplitude / annual mean')
plt.legend(ncol=4); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# step 8: interannual change of the constituent amplitudes
def annual(months=None, first=2020, last=2025):
    rows = []
    for yr in range(first, last+1):
        sel = h[h.index.year == yr]
        if months is not None:
            sel = sel[sel.index.month.isin(months)]
        if len(sel) < 2000:
            rows.append(dict(year=yr, n=len(sel)))
            continue
        Ai = amplitudes(fit(sel.index, sel.values))['amplitude_m']*100
        rows.append(dict(year=yr, n=len(sel), **{k: Ai[k] for k in ['M2','S2','K1','O1','M4','M6']}))
    return pd.DataFrame(rows).set_index('year')

def trend(D, k):
    d = D[k].dropna()
    x = d.index.values - d.index.values.mean()
    b, a = np.polyfit(x, d.values, 1)
    r = d.values - (a + b*x)
    se = np.sqrt((r @ r / (len(d)-2)) / (x @ x))
    return b, se

ALL = annual(); DRY = annual([12,1,2,3,4]); WET = annual([6,7,8,9,10])
print('all months (cm)'); print(ALL[['M2','S2','K1','O1','M4','M6']].round(2).to_string())
print()
for k in ['M2','S2','K1','O1','M4','M6']:
    b, se = trend(ALL, k)
    flag = '  significant' if abs(b/se) > 2.57 else ''
    print(f'  {k:>3}: {b:+7.3f} +/- {se:.3f} cm/yr  (t = {b/se:+5.2f}){flag}')

print('\nM2 trend by season')
for lab, D in (('all', ALL), ('dry Dec-Apr', DRY), ('wet Jun-Oct', WET)):
    b, se = trend(D, 'M2')
    print(f'  {lab:<12}: {b:+7.3f} +/- {se:.3f} cm/yr  (t = {b/se:+5.2f})')

plt.figure(figsize=(7,3.4))
for D, lab, st in ((ALL,'all months','o-'), (DRY,'dry season','s--'), (WET,'wet season','^:')):
    d = D['M2'].dropna(); plt.plot(d.index, d.values, st, label=lab)
plt.xlabel('year'); plt.ylabel('M2 amplitude (cm)')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# step 9: could the M2 decline be an artefact of the 18.6-year nodal cycle?
tt = pd.DatetimeIndex([pd.Timestamp(f'{y}-07-01') for y in range(2019, 2027)])
nd = nodal(tt); fM2, _ = nd['M2']
jd = tt.to_julian_date().values; T = (jd - 2451545.0)/36525.0
N = (125.04452 - 1934.136261*T) % 360

print(f'{"year":>6} {"N (deg)":>9} {"f_M2":>8}')
for i, y in enumerate(range(2019, 2027)):
    print(f'{y:>6} {N[i]:9.1f} {fM2[i]:8.4f}')

chg = 100*(fM2[6]-fM2[1])/fM2[1]
b, se = trend(ALL, 'M2')
mean_M2 = ALL['M2'].mean()
spurious = 0.10 * chg/100 * mean_M2 / 5      # 10 % error in the nodal formula, over 5 years
print(f'\nnode moved through {abs(N[6]-N[1]):.0f} deg of the 360 deg cycle in 2020-2025')
print(f'f_M2 changed by {chg:+.2f} % over the same interval')
print(f'a 10 % error in the nodal correction would leave {spurious:+.3f} cm/yr')
print(f'the observed trend is {b:+.3f} cm/yr, i.e. {abs(b/spurious):.0f} times larger')

In [ ]:
# step 10: daily mean water level, and the coverage of the record
# A day is kept only if it carries at least 18 hours of data. This daily series is
# what every Gaussian process in steps 14-19 is fitted to.
#
# A plain arithmetic mean is biased on a day that is missing hours, because the
# missing hours are not spread evenly over the tidal cycle: on the worst days here
# that bias reaches 30 cm. Removing the harmonic tide first, averaging the residual
# and adding the mean level back removes it.
pred_h = coef[0] + design(h.index) @ coef[1:]
res_h  = pd.Series(h.values - pred_h, index=h.index)
cnt    = res_h.resample('D').count()
fullD  = pd.date_range(hourly.index[0].normalize(), hourly.index[-1].normalize(), freq='D')
dm     = (res_h.resample('D').mean() + coef[0]).reindex(fullD)
cnt    = cnt.reindex(fullD).fillna(0).astype(int)
daily  = pd.DataFrame({'dmwl': dm.where(cnt >= 18), 'hours': cnt})
daily.index.name = 'date'

print(f'calendar days : {len(daily)}')
print(f'days kept     : {int(daily.dmwl.notna().sum())}')
print(f'days dropped  : {int(daily.dmwl.isna().sum())}')

obs_days = daily.index[daily.dmwl.notna()]
gapd = np.diff(obs_days.values).astype('timedelta64[D]').astype(int)
print('\ngaps longer than three days')
for i in np.where(gapd > 3)[0]:
    print(f'  {obs_days[i].date()} -> {obs_days[i+1].date()}   {gapd[i]-1} days missing')

mo = daily.hours.resample('MS').mean()
plt.figure(figsize=(11, 3))
plt.bar(mo.index, mo.values, width=22, color='tab:blue')
plt.axhline(18, color='tab:red', ls='--', label='18 h threshold')
plt.ylabel('hours of data per day'); plt.xlabel('year'); plt.ylim(0, 26)
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
# The three episodes that dominate the loss are a 25-day outage in November 2019,
# a 39-day outage from 26 November 2020 and a 75-day outage from 24 June 2021.
# The last removes the whole 2021 wet season, which is why no wet-season estimate
# is available for that year in step 12.

In [ ]:
# step 11: is the duration asymmetry changing from year to year?
# seq comes from step 5: the alternating sequence of high and low waters.
rt, rv, ft, fv = [], [], [], []
for a, b in zip(seq[:-1], seq[1:]):
    dt = (b[0] - a[0]).total_seconds()/3600.0
    if dt < 2 or dt > 20: continue
    if a[2] == 'L': rt.append(a[0]); rv.append(dt)
    else:           ft.append(a[0]); fv.append(dt)
R = pd.Series(rv, index=pd.DatetimeIndex(rt))
F = pd.Series(fv, index=pd.DatetimeIndex(ft))
print(f'rise n={len(R)} mean {R.mean():.2f} h    fall n={len(F)} mean {F.mean():.2f} h'
      f'    asymmetry {F.mean()-R.mean():+.2f} h')

rows = []
for y in range(2019, 2027):
    r, f = R[R.index.year == y], F[F.index.year == y]
    if len(r) < 50: continue
    rows.append(dict(year=y, n_rise=len(r), rise=r.mean(), se_rise=r.std(ddof=1)/np.sqrt(len(r)),
                     n_fall=len(f), fall=f.mean(), se_fall=f.std(ddof=1)/np.sqrt(len(f)),
                     asym=f.mean()-r.mean(),
                     se_asym=np.sqrt(r.var(ddof=1)/len(r) + f.var(ddof=1)/len(f))))
DUR = pd.DataFrame(rows).set_index('year')
print('\n' + DUR.round(3).to_string())

# weighted straight line through the complete years only (2019 and 2026 are partial)
full = DUR.loc[2020:2025]
x = full.index.values.astype(float); yv = full.asym.values; w = 1/full.se_asym.values**2
X = np.column_stack([np.ones(len(x)), x - x.mean()])
C = np.linalg.inv(X.T @ np.diag(w) @ X); bb = C @ (X.T @ np.diag(w) @ yv)
print(f'\ntrend in the duration asymmetry: {bb[1]:+.3f} +/- {np.sqrt(C[1,1]):.3f} h/yr'
      f'   (t = {bb[1]/np.sqrt(C[1,1]):.2f}) -- not distinguishable from no change')

for lab, mos in (('wet Jun-Oct', [6,7,8,9,10]), ('dry Dec-Mar', [12,1,2,3])):
    r, f = R[R.index.month.isin(mos)], F[F.index.month.isin(mos)]
    se = np.sqrt(r.var(ddof=1)/len(r) + f.var(ddof=1)/len(f))
    print(f'{lab}: asymmetry {f.mean()-r.mean():.3f} +/- {se:.3f} h')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].errorbar(DUR.index, DUR.fall, DUR.se_fall, marker='s', label='fall')
ax[0].errorbar(DUR.index, DUR.rise, DUR.se_rise, marker='o', label='rise')
ax[0].set_ylabel('hours'); ax[0].set_xlabel('year'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].errorbar(DUR.index, DUR.asym, DUR.se_asym, marker='o', color='k', ls='none')
ax[1].plot(x, bb[0] + bb[1]*(x - x.mean()), color='gray', ls='--')
ax[1].set_ylabel('fall - rise (h)'); ax[1].set_xlabel('year'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# step 12: non-linear distortion in the coordinates of Friedrichs and Aubrey (1988)
# The M4/M2 amplitude ratio measures how strong the distortion is; the relative
# phase 2*phiM2 - phiM4 says which way it leans. Between 0 and 180 deg is flood
# dominant, which should agree with the shorter rise found in step 11.
def distortion(idx):
    c = fit(idx, hourly.loc[idx].values)
    a = amplitudes(c).set_index('constituent')
    rel = (2*a.loc['M2','phase_deg_rel2020'] - a.loc['M4','phase_deg_rel2020']) % 360
    return a.loc['M2','amplitude_m'], a.loc['M4','amplitude_m'], \
           a.loc['M4','amplitude_m']/a.loc['M2','amplitude_m'], rel

m2a, m4a, ratio_all, rel_all = distortion(h.index)
print(f'whole record : M2 {100*m2a:.2f} cm   M4 {100*m4a:.2f} cm   '
      f'M4/M2 {ratio_all:.4f}   2phiM2-phiM4 {rel_all:.1f} deg')

rows = []
for y in range(2020, 2026):
    i = h.index[h.index.year == y]
    _, _, ra, re = distortion(i); rows.append(dict(period=str(y), n=len(i), ratio=ra, rel_phase=re))
for y in range(2020, 2026):
    for lab, mos in (('wet', [6,7,8,9,10]), ('dry', [12,1,2,3])):
        i = h.index[(h.index.year == y) & (h.index.month.isin(mos))]
        if len(i) < 2000: continue
        _, _, ra, re = distortion(i); rows.append(dict(period=f'{y} {lab}', n=len(i), ratio=ra, rel_phase=re))
DIS = pd.DataFrame(rows).set_index('period')
print('\n' + DIS.round(4).to_string())

print()
for lab, mos in (('wet pooled', [6,7,8,9,10]), ('dry pooled', [12,1,2,3])):
    i = h.index[h.index.month.isin(mos)]
    _, _, ra, re = distortion(i)
    print(f'{lab}: M4/M2 {ra:.4f}   relative phase {re:.1f} deg')
# The wet season carries the larger ratio in every year for which both seasons
# survive quality control. The 2021 wet season is missing (the 75-day outage).

yr = DIS.loc[[str(y) for y in range(2020, 2026)]]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].scatter(yr.rel_phase, yr.ratio)
for p, rw in yr.iterrows():
    ax[0].annotate(p, (rw.rel_phase, rw.ratio), fontsize=8, xytext=(3,3), textcoords='offset points')
ax[0].scatter([rel_all], [ratio_all], marker='*', s=140, color='k', label='whole record')
ax[0].set_title('year by year'); ax[0].legend(fontsize=8)
for y in range(2020, 2026):
    wk, dk = f'{y} wet', f'{y} dry'
    if wk in DIS.index and dk in DIS.index:
        ax[1].plot([DIS.loc[dk,'rel_phase'], DIS.loc[wk,'rel_phase']],
                   [DIS.loc[dk,'ratio'], DIS.loc[wk,'ratio']], color='gray', lw=.8, zorder=1)
wet_i = [i for i in DIS.index if i.endswith('wet')]; dry_i = [i for i in DIS.index if i.endswith('dry')]
ax[1].scatter(DIS.loc[wet_i,'rel_phase'], DIS.loc[wet_i,'ratio'], label='wet Jun-Oct', zorder=2)
ax[1].scatter(DIS.loc[dry_i,'rel_phase'], DIS.loc[dry_i,'ratio'], facecolors='none',
              edgecolors='tab:orange', label='dry Dec-Mar', zorder=2)
ax[1].set_title('wet against dry season'); ax[1].legend(fontsize=8)
for a_ in ax:
    a_.set_xlabel('2*phiM2 - phiM4 (deg)'); a_.set_ylabel('M4 / M2'); a_.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# step 13: the thirty-three constituents with moving-block bootstrap errors
# Ordinary least-squares standard errors would be far too small here because the
# residuals are strongly autocorrelated. Resampling the residuals in 30-day blocks
# keeps that correlation, and is the same device used for the seasonal intervals.
X33 = np.column_stack([np.ones(len(h)), design(h.index)])
fit0 = X33 @ coef
res33 = h.values - fit0
rng33 = np.random.default_rng(7)
L, B = 720, 200                       # 30-day blocks, 200 replicates
nb33 = int(np.ceil(len(h)/L))
A0 = amplitudes(coef)
amp_b = np.zeros((B, len(NAMES))); pha_b = np.zeros((B, len(NAMES)))
for b_ in range(B):
    st = rng33.integers(0, len(h)-L, size=nb33)
    rb = np.concatenate([res33[s0:s0+L] for s0 in st])[:len(h)]
    cb, *_ = np.linalg.lstsq(X33, fit0 + rb, rcond=None)
    Ab = amplitudes(cb)
    amp_b[b_] = Ab.amplitude_m.values
    pha_b[b_] = Ab.phase_deg_rel2020.values

wrap = ((pha_b - A0.phase_deg_rel2020.values + 180) % 360) - 180   # signed phase error
TAB33 = pd.DataFrame({'constituent': NAMES,
                      'speed_deg_h': [CONST[n][0] for n in NAMES],
                      'amp_cm':       100*A0.amplitude_m.values,
                      'amp_se_cm':    100*amp_b.std(0, ddof=1),
                      'phase_deg':    A0.phase_deg_rel2020.values,
                      'phase_se_deg': wrap.std(0, ddof=1)})
TAB33 = TAB33.sort_values('amp_cm', ascending=False).reset_index(drop=True)
print(TAB33.round(3).to_string(index=False))
print('\nA phase is only meaningful where its bootstrap spread is small; for M3,')
print('whose amplitude is about 1 mm, it is not.')

In [ ]:
# step 14: Gaussian process machinery
# Model: y(t) = h(t) beta + f(t) + e, with f a zero-mean Gaussian process.
# beta is profiled out by generalised least squares and the kernel hyperparameters
# are fitted by maximum marginal likelihood with analytic gradients
# (Xu et al. 2024, GPS Solutions 28:79). Time is in years from T0.
from scipy.optimize import minimize
from scipy.linalg import cho_factor, cho_solve

def tyears(idx): return ((idx - T0).days.values)/365.25

def basis(t, harmonics=(1,)):
    cols = [np.ones_like(t), t]
    for k in harmonics: cols += [np.sin(2*np.pi*k*t), np.cos(2*np.pi*k*t)]
    return np.column_stack(cols)

# Each kernel returns K and the derivatives of K with respect to the LOG parameters.
def _m12(r, lp):
    sf2, ell = np.exp(lp[0]), np.exp(lp[1]); K = sf2*np.exp(-r/ell)
    return K, [K, K*r/ell]
def _m32(r, lp):
    sf2, ell = np.exp(lp[0]), np.exp(lp[1]); a = np.sqrt(3)*r/ell; e = np.exp(-a)
    K = sf2*(1+a)*e
    return K, [K, sf2*a*a*e]
def _m52(r, lp):
    sf2, ell = np.exp(lp[0]), np.exp(lp[1]); a = np.sqrt(5)*r/ell; e = np.exp(-a)
    K = sf2*(1 + a + a*a/3.0)*e
    return K, [K, sf2*e*a*a*(1.0+a)/3.0]
def _rq(r, lp):
    sf2, ell, al = np.exp(lp[0]), np.exp(lp[1]), np.exp(lp[2])
    w = r*r/(2.0*al*ell*ell); u = 1.0 + w; K = sf2*u**(-al)
    return K, [K, 2.0*al*K*w/u, al*K*(w/u - np.log(u))]
def _qp(r, lp, P=1.0):
    sf2, l1, l2 = np.exp(lp[0]), np.exp(lp[1]), np.exp(lp[2]); s2 = np.sin(np.pi*r/P)**2
    K = sf2*np.exp(-2*s2/l1**2 - r**2/(2*l2**2))
    return K, [K, K*4*s2/l1**2, K*r**2/l2**2]
def _sum(*parts):
    def f(r, lp):
        K = 0.0; G = []; i = 0
        for fn, n in parts:
            k, g = fn(r, lp[i:i+n]); K = K + k; G += g; i += n
        return K, G
    return f

# name: (kernel, parameter names, starting log-parameters, harmonics in the mean basis)
KERNELS = {
 'GPM12':  (_m12, ['sf2','ell_yr'], [np.log(0.01), np.log(0.01)], (1,)),
 'GP':     (_m32, ['sf2','ell_yr'], [np.log(0.01), np.log(0.05)], (1,)),
 'GPM52':  (_m52, ['sf2','ell_yr'], [np.log(0.01), np.log(0.02)], (1,)),
 'GPRQ':   (_rq,  ['sf2','ell_yr','alpha'], [np.log(0.01), np.log(0.02), np.log(1.0)], (1,)),
 'GP2M32': (_sum((_m32,2),(_m32,2)), ['sf2_short','ell_short_yr','sf2_long','ell_long_yr'],
            [np.log(0.007), np.log(0.005), np.log(0.002), np.log(0.1)], (1,)),
 'GPQP':   (_sum((_m32,2),(_qp,3)), ['sf2_m32','ell_m32_yr','sf2_qp','lp_qp','ld_qp_yr'],
            [np.log(0.008), np.log(0.01), np.log(0.004), np.log(0.8), np.log(3.0)], (1,)),
}
# GP-2M-SA is the two-scale kernel with a semiannual term added to the mean basis.
KERNELS['GP2M32SA'] = (KERNELS['GP2M32'][0], KERNELS['GP2M32'][1], KERNELS['GP2M32'][2], (1,2))

VAR = ['GPM12','GP','GPM52','GPRQ','GP2M32','GP2M32SA','GPQP']
LABEL = {'GPM12':'Matern 1/2','GP':'Matern 3/2','GPM52':'Matern 5/2','GPRQ':'rational quadratic',
         'GP2M32':'two-scale M3/2','GP2M32SA':'GP-2M-SA','GPQP':'M3/2 + quasi-periodic'}

class GPModel:
    def __init__(self, kind='GP'):
        self.kind = kind; self.kf, self.pnames, self.p0, self.harm = KERNELS[kind]
    def _prep(self, lp, R):
        K, G = self.kf(R, lp[:-1]); sn2 = np.exp(lp[-1])
        K = K.copy(); K[np.diag_indices_from(K)] += sn2 + 1e-8
        return K, G + [sn2*np.eye(len(R))]
    def _nll(self, lp, y, H, R):
        K, G = self._prep(lp, R)
        try: c = cho_factor(K, lower=True)
        except np.linalg.LinAlgError: return 1e10, np.zeros_like(lp)
        KiH = cho_solve(c, H); Am = H.T @ KiH
        beta = np.linalg.solve(Am, KiH.T @ y); r = y - H @ beta
        alpha = cho_solve(c, r)
        nll = 0.5*r@alpha + np.log(np.diag(c[0])).sum() + 0.5*len(y)*np.log(2*np.pi)
        Ki = cho_solve(c, np.eye(len(y))); Wm = np.outer(alpha, alpha) - Ki
        return nll, np.array([-0.5*np.sum(Wm*g) for g in G])
    def fit(self, t, y):
        self.t, self.y = t, y
        H = basis(t, self.harm); R = np.abs(t[:,None] - t[None,:])
        o = minimize(self._nll, np.r_[self.p0, np.log(0.003)], args=(y, H, R), jac=True,
                     method='L-BFGS-B', bounds=[(-14, 4)]*(len(self.p0)+1))
        self.lp, self.nll = o.x, o.fun
        K, _ = self._prep(o.x, R); self.c = cho_factor(K, lower=True)
        KiH = cho_solve(self.c, H); self.Ainv = np.linalg.inv(H.T @ KiH)
        self.beta = self.Ainv @ (KiH.T @ y); self.alpha = cho_solve(self.c, y - H @ self.beta)
        self.KiH, self.H = KiH, H
        self.params = dict(zip(self.pnames + ['sn2'], np.exp(o.x)))
        self.k = len(o.x) + H.shape[1]
        return self
    def predict(self, ts):
        Hs = basis(ts, self.harm)
        Ks, _ = self.kf(np.abs(ts[:,None] - self.t[None,:]), self.lp[:-1])
        mean = Hs @ self.beta + Ks @ self.alpha
        kss, _ = self.kf(np.zeros(1), self.lp[:-1])
        V = cho_solve(self.c, Ks.T)
        var = np.atleast_1d(kss)[0] - np.einsum('ij,ji->i', Ks, V)
        Rm = Hs - Ks @ self.KiH                      # uncertainty in beta
        var = var + np.einsum('ij,jk,ik->i', Rm, self.Ainv, Rm)
        return mean, np.sqrt(np.maximum(var, 0))
    def rate(self):  return self.beta[1], np.sqrt(self.Ainv[1,1])
    def aic(self):   return 2*self.nll + 2*self.k

td = tyears(daily.index); yd = daily.dmwl.values
obs_d = ~np.isnan(yd)
print(f'daily series for the Gaussian process: {int(obs_d.sum())} observed days of {len(yd)}')

In [ ]:
# step 15: the seven covariance structures on the full record
# Leave-one-out residuals come from a single matrix inversion rather than n refits.
# With a vague prior on beta,  r_i = [Kt^-1 y]_i / [Kt^-1]_ii  with
# Kt = K_y + H Sigma_beta H'.  Using the profiled Ainv here instead of the vague
# prior gives the wrong answer, which is easy to do by accident.
# Runtime is a few minutes: each fit is O(n^3) on n = 2412 days.
import time
GPFIT, LOO_RES = {}, {}
rows = []
tobs, yobs = td[obs_d], yd[obs_d]
Robs = np.abs(tobs[:,None] - tobs[None,:])
for k in VAR:
    t0 = time.time()
    g = GPModel(k).fit(tobs, yobs)
    K, _ = g._prep(g.lp, Robs)
    Hb = basis(tobs, g.harm)
    Kt = K + Hb @ (1e4*np.eye(Hb.shape[1])) @ Hb.T      # vague prior on beta
    Kti = np.linalg.inv(Kt)
    r = (Kti @ yobs)/np.diag(Kti)
    LOO_RES[k] = r; GPFIT[k] = g
    rate, se = g.rate()
    rows.append(dict(kernel=LABEL[k], n_hyper=len(g.lp), n_basis=Hb.shape[1],
                     LOO_cm=100*np.sqrt(np.mean(r**2)), AIC=g.aic(),
                     rate_mm_yr=1000*rate, rate_se=1000*se, sec=time.time()-t0))
FULL = pd.DataFrame(rows, index=VAR)
print(FULL.round(3).to_string())
print('\nfitted hyperparameters')
for k in VAR:
    print(f'  {LABEL[k]:22s}', {a: round(b, 6) for a, b in GPFIT[k].params.items()})

In [ ]:
# step 16: are the leave-one-out differences real?
# The squared residuals are paired day by day against GP-2M-SA. They are serially
# correlated -- the autocorrelation of the differences still reaches about 0.3 at a
# lag of two days, a remnant of the spring-neap cycle -- so an ordinary paired t
# test overstates significance. Newey-West with a lag of eight days and a
# moving-block bootstrap both widen the standard error.
from scipy import stats

def newey_west(d, lag=8):
    n = len(d); dm = d - d.mean(); g0 = dm @ dm / n; s = g0
    for L_ in range(1, lag+1):
        g = dm[:-L_] @ dm[L_:] / n
        s += 2*(1 - L_/(lag+1))*g
    return np.sqrt(s/n)

def block_boot(d, L_=30, B_=2000, seed=1):
    rng = np.random.default_rng(seed); n = len(d); nb = int(np.ceil(n/L_)); out = np.empty(B_)
    for b_ in range(B_):
        st = rng.integers(0, n-L_, size=nb)
        out[b_] = np.concatenate([d[s0:s0+L_] for s0 in st])[:n].mean()
    return out.std(ddof=1)

ref = LOO_RES['GP2M32SA']
print(f'{"kernel":24s} {"mean diff":>11s} {"t naive":>8s} {"t HAC":>8s} {"t boot":>8s} {"p HAC":>10s}')
for k in VAR:
    if k == 'GP2M32SA': continue
    d = LOO_RES[k]**2 - ref**2
    t_n = d.mean()/(d.std(ddof=1)/np.sqrt(len(d)))
    t_h = d.mean()/newey_west(d)
    t_b = d.mean()/block_boot(d)
    p_h = 2*(1 - stats.norm.cdf(abs(t_h)))
    print(f'{LABEL[k]:24s} {1e4*d.mean():+11.4f} {t_n:8.2f} {t_h:8.2f} {t_b:8.2f} {p_h:10.2e}')
print('\n(mean diff is in cm^2, positive meaning the kernel is worse than GP-2M-SA)')

print('\nautocorrelation of the paired differences at lags 1, 2, 3, 5, 10')
for k in VAR:
    if k == 'GP2M32SA': continue
    dd = LOO_RES[k]**2 - ref**2; dm = dd - dd.mean()
    print(f'  {LABEL[k]:24s}',
          [round(float(np.corrcoef(dm[:-L_], dm[L_:])[0,1]), 3) for L_ in (1,2,3,5,10)])
print('The lag-two correlation is the large one -- it is a remnant of the spring-neap\n'
      'cycle -- which is why the Newey-West lag is set well beyond it.')

In [ ]:
# step 17: where the seven structures actually differ
# All seven fill a real gap with almost the same curve, so plotting them one panel
# each is uninformative. What separates them is (a) the shape of the fitted
# correlation function, (b) the predictive standard deviation through the gap and
# (c) the reconstructed mean relative to GP-2M-SA.
G0, G1 = pd.Timestamp('2020-11-26'), pd.Timestamp('2021-01-05')   # the 39-day gap
tsx = pd.date_range(G0 - pd.Timedelta('20D'), G1 + pd.Timedelta('20D'), freq='D')
tg = tyears(tsx)
PRED = {k: GPFIT[k].predict(tg) for k in VAR}
inside = (tsx >= G0) & (tsx <= G1)
refm = PRED['GP2M32SA'][0]

print(f'{"kernel":24s} {"mid-gap sd (cm)":>16s} {"max |mean - SA| in gap (cm)":>30s}')
for k in VAR:
    mu, sd = PRED[k]
    print(f'{LABEL[k]:24s} {100*sd[inside].max():16.2f} {100*np.abs(mu-refm)[inside].max():30.2f}')
print('\nThe filled values agree to within about 3 cm while the intervals differ by 17 %:')
print('the kernel changes the uncertainty far more than it changes the value.')

lags = np.linspace(0, 40, 401)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for k in VAR:
    g = GPFIT[k]
    Kv, _ = g.kf(lags/365.25, g.lp[:-1]); Kv = np.atleast_1d(Kv)
    ax[0].semilogy(lags, np.maximum(Kv/Kv[0], 2e-3), label=LABEL[k])
    mu, sd = PRED[k]
    ax[1].plot((tsx - G0).days, 100*sd)
    ax[2].plot((tsx - G0).days, 100*(mu - refm))
ax[0].set_xlabel('lag (days)'); ax[0].set_ylabel('k(tau)/k(0)'); ax[0].set_title('what the kernel assumes')
ax[1].set_xlabel('day into the gap'); ax[1].set_ylabel('predictive s.d. (cm)'); ax[1].set_title('how uncertain it is')
ax[2].set_xlabel('day into the gap'); ax[2].set_ylabel('mean - GP-2M-SA (cm)'); ax[2].set_title('how different the values are')
for a_ in ax: a_.grid(alpha=.3)
ax[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

plt.figure(figsize=(9, 3.6))
for k in VAR:
    plt.plot((tsx - G0).days, PRED[k][0], lw=2.4 if k == 'GP2M32SA' else 1, label=LABEL[k])
obs_win = daily.dmwl.reindex(tsx).dropna()
plt.plot((obs_win.index - G0).days, obs_win.values, 'k.', ms=4, label='observed')
plt.axvspan(0, (G1-G0).days, color='0.9', zorder=0)
plt.xlabel('days from 26 November 2020'); plt.ylabel('daily mean water level (m)')
plt.legend(fontsize=7, ncol=2); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# step 18: the 26 withheld-data experiments
# Days are removed from the fit in blocks, each structure is refitted on what
# remains, and the removed days are predicted. The design varies the size, the
# position and the shape of what is withheld:
#   A  one interior block, 4 sizes x 2 positions                      =  8
#   B  one block at the end, so the model must predict forward, 4 sizes =  4
#   C1 the 75-day summer window, in each year where it is complete    =  5
#   C2 the 39-day winter window, likewise                             =  5
#   D  runs of 1 to 10 days scattered through the record, 4 sizes     =  4
# The total of 26 follows from the design rather than from a target.
#
# WARNING: this is the slow cell. Seven fits per case at O(n^3), about 5-10 min
# per case, so roughly 3-4 hours in total on a Colab CPU. Set CASES below to a
# short list to try a few, or leave it as None to run all 26.
CASES = None

n_d = len(daily); idxd = daily.index
def make_cases():
    C = []
    for pct in (10, 20, 30, 40):
        Lb = int(round(n_d*pct/100))
        for cpos in (0.35, 0.65):
            s0 = int(n_d*cpos - Lb/2); msk = np.zeros(n_d, bool); msk[s0:s0+Lb] = True
            C.append(dict(group='A', label=f'{pct}% block at {int(cpos*100)}%', mask=msk))
    for pct in (10, 20, 30, 40):
        Lb = int(round(n_d*pct/100)); msk = np.zeros(n_d, bool); msk[n_d-Lb:] = True
        C.append(dict(group='B', label=f'last {pct}%', mask=msk))
    for y in (2020, 2022, 2023, 2024, 2025):
        C.append(dict(group='C1', label=f'{y}-06-25 to {y}-09-07',
                      mask=np.asarray((idxd >= f'{y}-06-25') & (idxd <= f'{y}-09-07'))))
    for y in (2021, 2022, 2023, 2024, 2025):
        C.append(dict(group='C2', label=f'{y}-11-27 to {y+1}-01-04',
                      mask=np.asarray((idxd >= f'{y}-11-27') & (idxd <= f'{y+1}-01-04'))))
    rng = np.random.default_rng(2024)
    for pct in (10, 20, 30, 40):
        msk = np.zeros(n_d, bool)
        while msk[obs_d].mean() < pct/100:
            Lb = rng.integers(1, 11); s0 = rng.integers(0, n_d-Lb); msk[s0:s0+Lb] = True
        C.append(dict(group='D', label=f'{pct}% scattered 1-10 d', mask=msk))
    return C

CS = make_cases()
print(f'{len(CS)} experiments:', pd.Series([c["group"] for c in CS]).value_counts().to_dict())

rows = []
todo = range(len(CS)) if CASES is None else CASES
for ci in todo:
    c = CS[ci]
    test = c['mask'] & obs_d; train = obs_d & ~c['mask']
    tt, yt = td[train], yd[train]; ts, ys = td[test], yd[test]
    t0 = time.time()
    for k in VAR:
        g = GPModel(k).fit(tt, yt)
        mu, sd = g.predict(ts)
        sdy = np.sqrt(sd**2 + g.params['sn2']); e = mu - ys
        rows.append(dict(case=ci, group=c['group'], label=c['label'], model=k,
                         RMSE_cm=100*np.sqrt(np.mean(e**2)), MAE_cm=100*np.mean(np.abs(e)),
                         bias_cm=100*np.mean(e), cover95=100*np.mean(np.abs(e) <= 1.96*sdy),
                         n_test=int(test.sum()), n_train=int(train.sum())))
    print(f'case {ci:2d} {c["group"]:3s} {time.time()-t0:6.1f}s  ' +
          '  '.join(f'{LABEL[r["model"]].split()[0]}:{r["RMSE_cm"]:.2f}' for r in rows[-7:]), flush=True)
EXP = pd.DataFrame(rows)
EXP.to_csv('withheld_experiments.csv', index=False)
print('\nsaved to withheld_experiments.csv')

In [ ]:
# step 19: what the 26 experiments say
# The unit of replication is the experiment, not the day: 26 cases, not 10276 days.
# Pairing is by case against GP-2M-SA.
P = EXP.pivot(index='case', columns='model', values='RMSE_cm')[VAR]
GRP = EXP.groupby('case').group.first()

by_group = P.groupby(GRP).mean()
by_group.loc['all 26'] = P.mean()
by_group.columns = [LABEL[k] for k in by_group.columns]
print('mean RMSE (cm) over the withheld days')
print(by_group.round(3).to_string())

print('\npaired comparison with GP-2M-SA over the 26 cases')
ref = P['GP2M32SA']
print(f'{"kernel":24s} {"delta":>8s} {"t":>7s} {"p_t":>8s} {"p_W":>8s} {"SA better in":>14s}')
for k in VAR:
    if k == 'GP2M32SA': continue
    d = (P[k] - ref).values
    t, p_t = stats.ttest_rel(P[k], ref)
    _, p_w = stats.wilcoxon(d)
    print(f'{LABEL[k]:24s} {d.mean():+8.3f} {t:7.2f} {p_t:8.3f} {p_w:8.3f} {int((d>0).sum()):10d} / 26')
print('\ndelta is the mean difference in RMSE, positive meaning GP-2M-SA is the better.')
print('The t test and the Wilcoxon test disagree because the differences are not')
print('symmetric: GP-2M-SA is a little ahead in about twenty cases and well behind')
print('in three of group C1, the 75-day summer gaps, where a seasonal term has to')
print('be carried across most of a season with nothing to anchor it.')

C = EXP.pivot(index='case', columns='model', values='cover95')[VAR]
cov = C.groupby(GRP).mean(); cov.loc['all 26'] = C.mean()
cov.columns = [LABEL[k] for k in cov.columns]
print('\ncoverage of the nominal 95 % predictive interval (%)')
print(cov.round(1).to_string())
print('All seven sit between 94 and 95 %, so none of them is over-confident:')
print('the narrower intervals are genuinely narrower, not merely optimistic.')

print('\nbest structure in each case')
print(P.idxmin(axis=1).map(LABEL).value_counts().to_string())